# Laboratorio #9 - Embeddings

**Juan Diego Letona Aguilar**

*20230285*

In [1]:
# Notebook 9 (el pdf dice laboratorio 8, pero corresponde al laboratorio 9).
# la primera parte (imports, carga de paquetes, etc.) es parecida a las iteraciones anteriores
# principalmente librerias y carga de paquetes necesarios para el proyecto

import sys
import subprocess

packages = [
    "pandas",
    "numpy",
    "nltk",
    "matplotlib",
    "scikit-learn",
    "scipy",
    "gensim",
    "torch",
    "tqdm"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", *packages
])

0

In [2]:
from pathlib import Path
from collections import Counter
import re
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm

from gensim.models import KeyedVectors, Word2Vec, FastText

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

#semilla para que los resultados sean reproducibles
SEED = 42
np.random.seed(SEED)

#recursos de NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Librerías cargadas correctamente.")

c:\Users\juanl\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerías cargadas correctamente.


## 1. Preparación de los corpus

**Carga de los corpus**

In [3]:
#cargamos el corpus de noticias (mismo dataset y misma limpieza basica de los laboratorios 1 y 2)

df = pd.read_csv("df_total.csv")

URL_COL = "url"
TEXT_COL = "news"
CATEGORY_COL = "Type"

corpus = df[[URL_COL, TEXT_COL, CATEGORY_COL]].copy()

corpus[TEXT_COL] = corpus[TEXT_COL].fillna("").astype(str)
corpus = corpus[corpus[TEXT_COL].str.strip().ne("")]
corpus = corpus.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

print("Dimensiones del DataFrame original:", df.shape)
print("Documentos de noticias despues de quitar vacios y duplicados:", len(corpus))

Dimensiones del DataFrame original: (1217, 3)
Documentos de noticias despues de quitar vacios y duplicados: 1137


In [4]:
#cargamos el Quijote (texto de Project Gutenberg)

with open("don-quijote.txt", encoding="utf-8-sig") as archivo:
    texto_quijote = archivo.read()

#quitamos la licencia y el encabezado de gutenberg, que no forman parte de la obra
inicio = texto_quijote.find("*** START")
fin = texto_quijote.find("*** END")
texto_quijote = texto_quijote[inicio:fin]
texto_quijote = texto_quijote.split("\n", 1)[1]

#cada capitulo es un documento
documentos_quijote = re.split(r"(?m)^\s*Cap[ií]tulo\s+", texto_quijote)
documentos_quijote = [doc for doc in documentos_quijote if doc.strip()]

print("Caracteres del Quijote:", f"{len(texto_quijote):,}")
print("Documentos del Quijote (capitulos y preliminares):", len(documentos_quijote))
print(documentos_quijote[1][:300])

Caracteres del Quijote: 2,098,169
Documentos del Quijote (capitulos y preliminares): 127
primero. Que trata de la condición y ejercicio del famoso hidalgo
don Quijote de la Mancha


En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho
tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua,
rocín flaco y galgo corredor. Una olla de algo más vaca que


**Segmentación en oraciones y tokenización**

Primero se arreglan algunos detalles del texto crudo. En las noticias hay lugares donde falta el espacio después del punto (por ejemplo "imaginamos.Para"), lo cual confunde al segmentador. En el Quijote el texto viene cortado en renglones, así que se unen antes de segmentar.

In [5]:
#funciones para segmentar en oraciones y tokenizar en palabras

patron_palabras = re.compile(r"[a-záéíóúüñ]+")

def segmentar_oraciones(texto):
    #agregamos el espacio que falta despues de un punto pegado a una mayuscula
    texto = re.sub(r"([a-záéíóúñ])([.!?])([A-ZÁÉÍÓÚÑ¿¡])", r"\1\2 \3", texto)
    #unimos los renglones y los espacios multiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return sent_tokenize(texto, language="spanish")

def tokenizar(oracion):
    #minusculas y solo palabras (letras con tilde, ñ y ü), sin puntuacion ni numeros
    return patron_palabras.findall(oracion.lower())

def preparar_corpus(documentos):
    #devuelve una lista de listas de tokens (una lista por oracion) y el numero de oraciones por documento
    oraciones_tokens = []
    oraciones_por_doc = []

    for documento in documentos:
        tokens_doc = [tokenizar(oracion) for oracion in segmentar_oraciones(documento)]
        #descartamos oraciones con menos de 2 tokens porque no tienen contexto
        tokens_doc = [oracion for oracion in tokens_doc if len(oracion) >= 2]
        oraciones_tokens.extend(tokens_doc)
        oraciones_por_doc.append(len(tokens_doc))

    return oraciones_tokens, oraciones_por_doc

oraciones_noticias, oraciones_por_doc_noticias = preparar_corpus(
    tqdm(corpus[TEXT_COL], desc="Noticias")
)
oraciones_quijote, oraciones_por_doc_quijote = preparar_corpus(
    tqdm(documentos_quijote, desc="Quijote")
)

print("Ejemplo de oracion de noticias:")
print(oraciones_noticias[10])
print("\nEjemplo de oracion del Quijote:")
print(oraciones_quijote[10])

Quijote: 100%|██████████| 127/127 [00:01<00:00, 103.79it/s]

Ejemplo de oracion de noticias:
['la', 'comisión', 'reguladora', 'de', 'valores', 'de', 'china', 'csrc', 'dijo', 'en', 'un', 'comunicado', 'que', 'tomó', 'nota', 'de', 'los', 'nuevos', 'requisitos', 'de', 'la', 'comisión', 'de', 'valores', 'de', 'estados', 'unidos', 'sec', 'para', 'las', 'empresas', 'chinas', 'y', 'que', 'ambas', 'partes', 'deberían', 'mantener', 'el', 'espíritu', 'de', 'respeto', 'mutuo', 'y', 'reforzar', 'las', 'comunicaciones', 'sobre', 'la', 'regulación']

Ejemplo de oracion del Quijote:
['la', 'cual', 'dicha', 'pena', 'sea', 'la', 'tercia', 'parte', 'para', 'la', 'persona', 'que', 'lo', 'acusare', 'y', 'la', 'otra', 'tercia', 'parte', 'para', 'nuestra', 'cámara', 'y', 'la', 'otra', 'tercia', 'parte', 'para', 'el', 'juez', 'que', 'lo', 'sentenciare']


**Normalización: decisiones y compatibilidad con fastText SUC**

In [6]:
#cargamos los vectores preentrenados de fastText SUC (tamaño M, 100 dimensiones)
#solo las 200,000 palabras mas frecuentes para no llenar la memoria

RUTA_SUC = Path("embeddings-m-model.vec")
URL_SUC = "https://zenodo.org/record/3234051/files/embeddings-m-model.vec?download=1"

if not RUTA_SUC.exists():
    print("Descargando embeddings-m-model.vec (1.1 GB), esto tarda un rato...")
    urllib.request.urlretrieve(URL_SUC, RUTA_SUC)

suc = KeyedVectors.load_word2vec_format(str(RUTA_SUC), limit=200_000)

print("Palabras en suc:", len(suc.key_to_index))
print("Dimensiones:", suc.vector_size)

Palabras en suc: 200000
Dimensiones: 100


In [7]:
#revisamos si suc distingue mayusculas y tildes

pruebas = ["españa", "España", "ESPAÑA", "espana", "méxico", "mexico", "Mexico", "también", "tambien", "gobierno", "Gobierno"]

revision_suc = pd.DataFrame({
    "palabra": pruebas,
    "esta_en_suc": [palabra in suc.key_to_index for palabra in pruebas],
    "posicion_en_vocabulario": [suc.key_to_index.get(palabra) for palabra in pruebas]
})

display(
    revision_suc.style
    .hide(axis="index")
    .set_caption("Revisión del vocabulario de SUC")
)

#cuantos tokens de cada corpus (ya normalizados) existen en suc
for nombre, oraciones in [("Noticias", oraciones_noticias), ("Quijote", oraciones_quijote)]:
    tokens = [token for oracion in oraciones for token in oracion]
    cobertura = np.mean([token in suc.key_to_index for token in tokens]) * 100
    print(f"{nombre}: {cobertura:.2f}% de los tokens estan en suc")

palabra,esta_en_suc,posicion_en_vocabulario
españa,True,541.000000
España,False,nan
ESPAÑA,False,nan
espana,True,67448.000000
méxico,True,797.000000
mexico,True,16941.000000
Mexico,False,nan
también,True,64.000000
tambien,True,4259.000000
gobierno,True,236.000000


Noticias: 98.79% de los tokens estan en suc
Quijote: 97.38% de los tokens estan en suc


## Decisiones de normalización

- Minúsculas: sí. SUC está completamente en minúsculas: "españa" está en el vocabulario (posición 541), pero "España" y "ESPAÑA" no. Si dejamos las mayúsculas, nuestras palabras no se podrían buscar en SUC y, además, "Gobierno" y "gobierno" serían dos tipos distintos con la mitad de contextos cada uno.
- Tildes: se conservan. SUC sí las distingue: "méxico" está en la posición 797 y "mexico" en la 16,941, es decir, la versión sin tilde es una variante poco frecuente (casi un error de escritura). Quitar tildes nos dejaría con formas poco comunes en SUC y, además, mezclaría palabras distintas ("él"/"el", "sí"/"si", "más"/"mas").
- Puntuación y números: se eliminan. Con la expresión regular solo quedan palabras (letras, tildes, ñ y ü). Los números aportan poco a los vecinos semánticos y llenarían el vocabulario de tipos muy raros.
- Compatibilidad con SUC: con esta normalización, el 98.79 % de los tokens de las noticias y el 97.38 % de los del Quijote existen en SUC, por lo que la comparación de las secciones 5 y 8 es justa.

## ¿Vale la pena quitar stopwords o lematizar antes de entrenar embeddings?

En este caso, no.

- Stopwords: Word2Vec cuenta posiciones, no palabras "importantes". Si quitamos "de", "la" o "que", palabras que estaban separadas por dos o tres posiciones quedan juntas y la ventana de contexto se deforma. Además, las stopwords dan información sintáctica (qué tipo de palabra suele ir antes o después) y Word2Vec ya trae el parámetro `sample` para submuestrear las palabras muy frecuentes sin destruir la oración.
- Lematización: en los laboratorios 1 y 2 servía para reducir el vocabulario en BoW y TF-IDF. Aquí perdería información porque si "gobiernos" y "gobernó" se colapsan en una sola forma, se pierde la diferencia entre sustantivo y verbo, entre plural y singular y entre pasado y presente, y el modelo ya no podría aprender que "gobernó" se parece a otros verbos en pasado. Y SUC no está lematizado, así que perderíamos compatibilidad.